
# Aztec `RewardBooster` Miss-Penalty Simulator

This notebook provides an interactive simulation for three `RewardBooster` parameter sets:

1. **Current / original**
2. **AZIP-5 proposed**
3. **htimsk softer revision**

It models what happens when a prover misses one or more epochs, how long it takes to return to full reward weight, the reward opportunity cost, the gross cash required to fund the recovery proofs, and the profitability line under configurable market and cost assumptions.

The notebook is intentionally self-contained. Change the controls in the interactive sections to test different ETH/USD, gas price, AZTC/USD, infrastructure cost, gas units per proof, active prover count, reward pool, and parameter values.



## Model assumptions

The simulator uses the following mechanics:

```text
score_after_success = min(maxScore, max(0, priorScore - decayRate * elapsedEpochs) + increment)
share = max(k - a * (maxScore - score)^2 / 1e10, minimum)
```

For a prover that was previously at `maxScore`, misses `m` consecutive epochs, and then submits a valid proof, the elapsed interval into the comeback proof is modeled as `m + 1` epochs. This matches the usual interpretation that a prover proves epoch `E`, misses epochs `E+1 ... E+m`, and returns at epoch `E+m+1`.

Economic model:

```text
gas_cost_usd = gas_units_per_proof * gas_price_gwei * 1e-9 * ETH_USD
cash_cost_per_proof = gas_cost_usd + infrastructure_cost_per_proof
reward_fraction = prover_share / (peer_full_weight_shares + prover_share)
reward_usd = epoch_reward_pool_AZTC * AZTC_USD * reward_fraction
profit_usd = reward_usd - cash_cost_per_proof
```

Important interpretation:

- **Reward opportunity cost** is the missed and reduced rewards relative to being a full-weight prover.
- **Gross recovery cash spend** is the cash needed to fund proofs until the first full-weight proof is reached.
- For a committed prover who would have continued proving anyway, the incremental economic burden is mostly reward opportunity cost. For an opportunistic prover, the gross cash needed to re-establish full weight is also important.


In [23]:

from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Dict, Tuple

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception as exc:
    WIDGETS_AVAILABLE = False
    widgets = None
    print("ipywidgets is not available in this environment. The non-interactive functions still work.")
    print(f"Import error: {exc}")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


In [24]:

@dataclass(frozen=True)
class RewardBoosterParams:
    name: str
    increment: float
    max_score: float
    a: float
    minimum: float
    k: float = 1_000_000.0
    decay_rate: float = 100_000.0


PARAMETER_PRESETS: Dict[str, RewardBoosterParams] = {
    "Current / original": RewardBoosterParams(
        name="Current / original",
        increment=125_000,
        max_score=15_000_000,
        a=1_000,
        minimum=100_000,
        k=1_000_000,
        decay_rate=100_000,
    ),
    "AZIP-5 proposed": RewardBoosterParams(
        name="AZIP-5 proposed",
        increment=101_400,
        max_score=367_500,
        a=250_000,
        minimum=10_000,
        k=1_000_000,
        decay_rate=100_000,
    ),
    "htimsk softer revision": RewardBoosterParams(
        name="htimsk softer revision",
        increment=110_000,
        max_score=367_500,
        a=100_000,
        minimum=10_000,
        k=1_000_000,
        decay_rate=100_000,
    ),
}

DEFAULT_GAS_UNITS_PER_PROOF = 3_850_000  # Replace with a fresher chain estimate if desired.
DEFAULT_INFRA_COST_PER_PROOF_USD = 78.0
DEFAULT_ETH_USD = 2_200.0
DEFAULT_GAS_PRICE_GWEI = 2.0
DEFAULT_AZTC_USD = 0.015
DEFAULT_EPOCH_REWARD_POOL_AZTC = 4_800.0
DEFAULT_ACTIVE_FULL_WEIGHT_PROVERS = 10
DEFAULT_EPOCH_MINUTES = 38

pd.DataFrame([asdict(p) for p in PARAMETER_PRESETS.values()])


,name,increment,max_score,a,minimum,k,decay_rate
0,Current / original,125000,15000000,1000,100000,1000000,100000
1,AZIP-5 proposed,101400,367500,250000,10000,1000000,100000
2,htimsk softer revision,110000,367500,100000,10000,1000000,100000



## Core simulation functions

These functions are used by both the static tables and the interactive dashboard. You can reuse them in your own analysis cells.


In [25]:

def gas_cost_usd(
    gas_units_per_proof: float,
    gas_price_gwei: float,
    eth_usd: float,
) -> float:
    """Return gas cost per proof in USD."""
    return gas_units_per_proof * gas_price_gwei * 1e-9 * eth_usd


def cash_cost_per_proof_usd(
    gas_units_per_proof: float,
    gas_price_gwei: float,
    eth_usd: float,
    infra_cost_per_proof_usd: float,
) -> float:
    return gas_cost_usd(gas_units_per_proof, gas_price_gwei, eth_usd) + infra_cost_per_proof_usd


def maintenance_coverage(params: RewardBoosterParams) -> float:
    """Long-run minimum coverage required for expected score drift >= 0."""
    if params.increment <= 0:
        return math.inf
    return min(1.0, params.decay_rate / params.increment)


def share_from_score(score: float | np.ndarray, params: RewardBoosterParams) -> float | np.ndarray:
    """Map score to RewardBooster share, with score capped at maxScore for modeling."""
    score_arr = np.asarray(score, dtype=float)
    capped_score = np.minimum(np.maximum(score_arr, 0), params.max_score)
    raw_share = params.k - params.a * (params.max_score - capped_score) ** 2 / 1e10
    shares = np.maximum(raw_share, params.minimum)
    if np.ndim(score) == 0:
        return float(shares)
    return shares


def successful_update(prior_score: float, elapsed_epochs: int, params: RewardBoosterParams) -> float:
    """Apply decay over elapsed epochs, add increment for a successful proof, and cap at maxScore."""
    decayed = max(0.0, prior_score - params.decay_rate * elapsed_epochs)
    return min(params.max_score, decayed + params.increment)


def reward_fraction_of_epoch_pool(
    prover_share: float,
    params: RewardBoosterParams,
    active_full_weight_provers: int,
) -> float:
    """
    Return the fraction of the epoch reward pool paid to this prover.

    active_full_weight_provers is the number of provers in the peer set if this prover were healthy,
    including this prover. Other provers are modeled as full weight.
    """
    active_full_weight_provers = max(int(active_full_weight_provers), 1)
    peer_count = max(active_full_weight_provers - 1, 0)
    denominator = peer_count * params.k + prover_share
    if denominator <= 0:
        return 0.0
    return prover_share / denominator


def healthy_reward_fraction(params: RewardBoosterParams, active_full_weight_provers: int) -> float:
    """The prover's reward fraction at full weight, assuming equal full-weight peers."""
    active_full_weight_provers = max(int(active_full_weight_provers), 1)
    return 1.0 / active_full_weight_provers


def recovery_dataframe(
    params: RewardBoosterParams,
    missed_epochs: int = 1,
    start_score: float | None = None,
    max_successful_proofs: int = 50_000,
) -> pd.DataFrame:
    """
    Simulate the path after missing `missed_epochs` consecutive epochs from `start_score`.

    Returns rows for missed epochs and subsequent successful proof submissions until the first full-weight proof.
    """
    if missed_epochs < 0:
        raise ValueError("missed_epochs must be non-negative")
    if start_score is None:
        start_score = params.max_score

    rows = []

    # Missed epochs: no proof, no reward, no cash spend modeled.
    for i in range(1, missed_epochs + 1):
        rows.append({
            "event": "missed_epoch",
            "timeline_epoch": i,
            "proof_index_after_return": np.nan,
            "elapsed_epochs_into_update": np.nan,
            "score": np.nan,
            "share": 0.0,
            "relative_weight": 0.0,
            "at_full_weight": False,
            "raw_weight_loss_eq": 1.0,
        })

    first_elapsed = 1 if missed_epochs == 0 else missed_epochs + 1
    score = successful_update(start_score, first_elapsed, params)

    for proof_idx in range(1, max_successful_proofs + 1):
        share = share_from_score(score, params)
        at_full_weight = score >= params.max_score - 1e-9
        rows.append({
            "event": "recovery_proof",
            "timeline_epoch": missed_epochs + proof_idx,
            "proof_index_after_return": proof_idx,
            "elapsed_epochs_into_update": first_elapsed if proof_idx == 1 else 1,
            "score": score,
            "share": share,
            "relative_weight": share / params.k,
            "at_full_weight": at_full_weight,
            "raw_weight_loss_eq": max(0.0, 1.0 - share / params.k),
        })
        if at_full_weight:
            break
        score = successful_update(score, 1, params)
    else:
        raise RuntimeError("Recovery did not reach full weight before max_successful_proofs.")

    return pd.DataFrame(rows)


def add_economics(
    df: pd.DataFrame,
    params: RewardBoosterParams,
    eth_usd: float = DEFAULT_ETH_USD,
    gas_price_gwei: float = DEFAULT_GAS_PRICE_GWEI,
    aztc_usd: float = DEFAULT_AZTC_USD,
    infra_cost_per_proof_usd: float = DEFAULT_INFRA_COST_PER_PROOF_USD,
    gas_units_per_proof: float = DEFAULT_GAS_UNITS_PER_PROOF,
    epoch_reward_pool_aztc: float = DEFAULT_EPOCH_REWARD_POOL_AZTC,
    active_full_weight_provers: int = DEFAULT_ACTIVE_FULL_WEIGHT_PROVERS,
) -> pd.DataFrame:
    """Add USD-denominated cost/revenue/profit columns to a recovery dataframe."""
    out = df.copy()
    gas_usd = gas_cost_usd(gas_units_per_proof, gas_price_gwei, eth_usd)
    cash_usd = gas_usd + infra_cost_per_proof_usd
    epoch_pool_usd = epoch_reward_pool_aztc * aztc_usd
    healthy_fraction = healthy_reward_fraction(params, active_full_weight_provers)
    healthy_reward_usd = epoch_pool_usd * healthy_fraction

    reward_fractions = []
    reward_usd_values = []
    cash_spend_values = []
    gas_spend_values = []
    infra_spend_values = []
    profit_values = []
    opportunity_loss_usd_values = []
    exact_opportunity_loss_eq_values = []

    for _, row in out.iterrows():
        if row["event"] == "missed_epoch":
            reward_fraction = 0.0
            reward_usd = 0.0
            gas_spend = 0.0
            infra_spend = 0.0
            cash_spend = 0.0
            profit_usd = 0.0
            opportunity_loss_usd = healthy_reward_usd
        else:
            reward_fraction = reward_fraction_of_epoch_pool(row["share"], params, active_full_weight_provers)
            reward_usd = epoch_pool_usd * reward_fraction
            gas_spend = gas_usd
            infra_spend = infra_cost_per_proof_usd
            cash_spend = cash_usd
            profit_usd = reward_usd - cash_spend
            opportunity_loss_usd = max(0.0, healthy_reward_usd - reward_usd)

        reward_fractions.append(reward_fraction)
        reward_usd_values.append(reward_usd)
        gas_spend_values.append(gas_spend)
        infra_spend_values.append(infra_spend)
        cash_spend_values.append(cash_spend)
        profit_values.append(profit_usd)
        opportunity_loss_usd_values.append(opportunity_loss_usd)
        exact_opportunity_loss_eq_values.append(
            opportunity_loss_usd / healthy_reward_usd if healthy_reward_usd > 0 else np.nan
        )

    out["reward_fraction_of_pool"] = reward_fractions
    out["reward_usd"] = reward_usd_values
    out["gas_spend_usd"] = gas_spend_values
    out["infra_spend_usd"] = infra_spend_values
    out["cash_spend_usd"] = cash_spend_values
    out["profit_usd"] = profit_values
    out["reward_opportunity_loss_usd"] = opportunity_loss_usd_values
    out["exact_opportunity_loss_eq"] = exact_opportunity_loss_eq_values
    out["cumulative_cash_spend_usd"] = out["cash_spend_usd"].cumsum()
    out["cumulative_reward_opportunity_loss_usd"] = out["reward_opportunity_loss_usd"].cumsum()
    out["cumulative_raw_weight_loss_eq"] = out["raw_weight_loss_eq"].cumsum()
    out["cumulative_exact_opportunity_loss_eq"] = out["exact_opportunity_loss_eq"].cumsum()
    return out


def break_even_share(
    params: RewardBoosterParams,
    cash_cost_usd_value: float,
    epoch_reward_pool_aztc: float,
    aztc_usd: float,
    active_full_weight_provers: int,
) -> Tuple[float, float, str]:
    """
    Return the share required to break even, relative weight, and a status string.

    The break-even equation is:
        epoch_pool_usd * share / ((N-1)*k + share) = cash_cost_usd
    """
    epoch_pool_usd = epoch_reward_pool_aztc * aztc_usd
    if epoch_pool_usd <= 0:
        return np.inf, np.inf, "No rewards: AZTC price or reward pool is zero."

    active_full_weight_provers = max(int(active_full_weight_provers), 1)
    peers = max(active_full_weight_provers - 1, 0)
    required_fraction = cash_cost_usd_value / epoch_pool_usd
    max_fraction = 1.0 / active_full_weight_provers

    if required_fraction <= 0:
        return 0.0, 0.0, "Profitable at any positive reward share."
    if required_fraction > max_fraction + 1e-15:
        return np.inf, np.inf, "Not profitable even at full weight."
    if peers == 0:
        return 0.0, 0.0, "Single-prover case: any positive share receives the whole pool."

    # share / (peers*k + share) = required_fraction
    share_required = (required_fraction * peers * params.k) / (1.0 - required_fraction)
    relative_weight_required = share_required / params.k
    return share_required, relative_weight_required, "Break-even share found."


def score_for_share(target_share: float, params: RewardBoosterParams) -> float:
    """Invert the share function to estimate the minimum score needed for `target_share`."""
    if not np.isfinite(target_share):
        return np.inf
    target_share = max(float(target_share), params.minimum)
    if target_share <= params.minimum:
        return 0.0
    if target_share >= params.k:
        return params.max_score
    delta = math.sqrt(max(0.0, (params.k - target_share) * 1e10 / params.a))
    return max(0.0, params.max_score - delta)


def break_even_aztc_price_at_full_weight(
    cash_cost_usd_value: float,
    epoch_reward_pool_aztc: float,
    active_full_weight_provers: int,
) -> float:
    """AZTC/USD needed for a full-weight prover to break even per proof."""
    if epoch_reward_pool_aztc <= 0:
        return np.inf
    return cash_cost_usd_value * max(int(active_full_weight_provers), 1) / epoch_reward_pool_aztc


def break_even_gas_price_at_full_weight_gwei(
    eth_usd: float,
    aztc_usd: float,
    epoch_reward_pool_aztc: float,
    active_full_weight_provers: int,
    gas_units_per_proof: float,
    infra_cost_per_proof_usd: float,
) -> float:
    """Gas price in gwei at which a full-weight proof breaks even."""
    reward_usd_full_weight = epoch_reward_pool_aztc * aztc_usd / max(int(active_full_weight_provers), 1)
    remaining_for_gas = reward_usd_full_weight - infra_cost_per_proof_usd
    if remaining_for_gas < 0 or eth_usd <= 0 or gas_units_per_proof <= 0:
        return -np.inf
    return remaining_for_gas / (gas_units_per_proof * 1e-9 * eth_usd)


def summarize_scenario(
    params: RewardBoosterParams,
    missed_epochs: int = 1,
    eth_usd: float = DEFAULT_ETH_USD,
    gas_price_gwei: float = DEFAULT_GAS_PRICE_GWEI,
    aztc_usd: float = DEFAULT_AZTC_USD,
    infra_cost_per_proof_usd: float = DEFAULT_INFRA_COST_PER_PROOF_USD,
    gas_units_per_proof: float = DEFAULT_GAS_UNITS_PER_PROOF,
    epoch_reward_pool_aztc: float = DEFAULT_EPOCH_REWARD_POOL_AZTC,
    active_full_weight_provers: int = DEFAULT_ACTIVE_FULL_WEIGHT_PROVERS,
) -> dict:
    """Return a compact summary of a miss/recovery scenario."""
    df = recovery_dataframe(params, missed_epochs=missed_epochs)
    econ = add_economics(
        df,
        params,
        eth_usd=eth_usd,
        gas_price_gwei=gas_price_gwei,
        aztc_usd=aztc_usd,
        infra_cost_per_proof_usd=infra_cost_per_proof_usd,
        gas_units_per_proof=gas_units_per_proof,
        epoch_reward_pool_aztc=epoch_reward_pool_aztc,
        active_full_weight_provers=active_full_weight_provers,
    )
    proofs = econ[econ["event"] == "recovery_proof"].copy()
    first_proof = proofs.iloc[0]
    underweight_proofs = int((proofs["share"] < params.k - 1e-9).sum())
    recovery_proofs = int(len(proofs))
    cash_per_proof = cash_cost_per_proof_usd(
        gas_units_per_proof, gas_price_gwei, eth_usd, infra_cost_per_proof_usd
    )
    share_required, rel_required, breakeven_status = break_even_share(
        params,
        cash_per_proof,
        epoch_reward_pool_aztc,
        aztc_usd,
        active_full_weight_provers,
    )
    score_required = score_for_share(share_required, params)

    return {
        "parameter_set": params.name,
        "missed_epochs": missed_epochs,
        "increment": params.increment,
        "maxScore": params.max_score,
        "a": params.a,
        "minimum": params.minimum,
        "maintenance_coverage_%": 100 * maintenance_coverage(params),
        "max_miss_rate_without_score_decay_%": 100 * max(0.0, 1.0 - maintenance_coverage(params)),
        "first_return_score": float(first_proof["score"]),
        "first_return_share_%": 100 * float(first_proof["share"] / params.k),
        "underweight_recovery_proofs": underweight_proofs,
        "recovery_proofs_to_full_weight": recovery_proofs,
        "recovery_time_hours": recovery_proofs * DEFAULT_EPOCH_MINUTES / 60,
        "raw_reward_weight_loss_eq": float(econ["raw_weight_loss_eq"].sum()),
        "exact_reward_opportunity_loss_eq": float(econ["exact_opportunity_loss_eq"].sum()),
        "reward_opportunity_loss_usd": float(econ["reward_opportunity_loss_usd"].sum()),
        "gross_recovery_cash_spend_usd": float(proofs["cash_spend_usd"].sum()),
        "gas_cost_per_proof_usd": gas_cost_usd(gas_units_per_proof, gas_price_gwei, eth_usd),
        "cash_cost_per_proof_usd": cash_per_proof,
        "full_weight_reward_per_proof_usd": epoch_reward_pool_aztc * aztc_usd / max(active_full_weight_provers, 1),
        "full_weight_profit_per_proof_usd": epoch_reward_pool_aztc * aztc_usd / max(active_full_weight_provers, 1) - cash_per_proof,
        "break_even_relative_weight_%": 100 * rel_required if np.isfinite(rel_required) else np.inf,
        "break_even_score": score_required,
        "break_even_aztc_usd_at_full_weight": break_even_aztc_price_at_full_weight(
            cash_per_proof, epoch_reward_pool_aztc, active_full_weight_provers
        ),
        "break_even_gas_gwei_at_full_weight": break_even_gas_price_at_full_weight_gwei(
            eth_usd,
            aztc_usd,
            epoch_reward_pool_aztc,
            active_full_weight_provers,
            gas_units_per_proof,
            infra_cost_per_proof_usd,
        ),
        "break_even_status": breakeven_status,
    }


def compare_presets_table(
    missed_epochs: int = 1,
    eth_usd: float = DEFAULT_ETH_USD,
    gas_price_gwei: float = DEFAULT_GAS_PRICE_GWEI,
    aztc_usd: float = DEFAULT_AZTC_USD,
    infra_cost_per_proof_usd: float = DEFAULT_INFRA_COST_PER_PROOF_USD,
    gas_units_per_proof: float = DEFAULT_GAS_UNITS_PER_PROOF,
    epoch_reward_pool_aztc: float = DEFAULT_EPOCH_REWARD_POOL_AZTC,
    active_full_weight_provers: int = DEFAULT_ACTIVE_FULL_WEIGHT_PROVERS,
) -> pd.DataFrame:
    rows = [
        summarize_scenario(
            p,
            missed_epochs=missed_epochs,
            eth_usd=eth_usd,
            gas_price_gwei=gas_price_gwei,
            aztc_usd=aztc_usd,
            infra_cost_per_proof_usd=infra_cost_per_proof_usd,
            gas_units_per_proof=gas_units_per_proof,
            epoch_reward_pool_aztc=epoch_reward_pool_aztc,
            active_full_weight_provers=active_full_weight_provers,
        )
        for p in PARAMETER_PRESETS.values()
    ]
    return pd.DataFrame(rows)



## Static comparison: one missed epoch with default economics

The defaults below are illustrative and should be overwritten with current network values when doing production analysis.


In [26]:

def format_summary_table(df: pd.DataFrame) -> pd.DataFrame:
    keep = [
        "parameter_set",
        "maintenance_coverage_%",
        "first_return_share_%",
        "underweight_recovery_proofs",
        "recovery_proofs_to_full_weight",
        "recovery_time_hours",
        "raw_reward_weight_loss_eq",
        "reward_opportunity_loss_usd",
        "gross_recovery_cash_spend_usd",
        "cash_cost_per_proof_usd",
        "full_weight_profit_per_proof_usd",
        "break_even_relative_weight_%",
        "break_even_aztc_usd_at_full_weight",
        "break_even_gas_gwei_at_full_weight",
        "break_even_status",
    ]
    return df[keep].copy()

static_summary = compare_presets_table(missed_epochs=1)
format_summary_table(static_summary)


,parameter_set,maintenance_coverage_%,first_return_share_%,underweight_recovery_proofs,recovery_proofs_to_full_weight,recovery_time_hours,raw_reward_weight_loss_eq,reward_opportunity_loss_usd,gross_recovery_cash_spend_usd,cash_cost_per_proof_usd,full_weight_profit_per_proof_usd,break_even_relative_weight_%,break_even_aztc_usd_at_full_weight,break_even_gas_gwei_at_full_weight,break_even_status
0,Current / original,80.000000,99.943750,3,4,2.533333,1.000875,7.205670,379.760000,94.940000,-87.740000,inf,0.197792,-inf,Not profitable even at full weight.
1,AZIP-5 proposed,98.619329,75.695100,71,72,45.600000,6.827964,45.533742,"6,835.680000",94.940000,-87.740000,inf,0.197792,-inf,Not profitable even at full weight.
2,htimsk softer revision,90.909091,91.900000,9,10,6.333333,1.285000,9.056800,949.400000,94.940000,-87.740000,inf,0.197792,-inf,Not profitable even at full weight.



## Interactive comparison dashboard

Use this dashboard to compare the three parameter sets side-by-side under the same economic assumptions.

The summary table includes:

- **`raw_reward_weight_loss_eq`**: missed and reduced share in simple max-weight epoch equivalents.
- **`reward_opportunity_loss_usd`**: exact pro-rata reward opportunity cost under the configured active prover count.
- **`gross_recovery_cash_spend_usd`**: gross cash required to fund all recovery proofs until the first full-weight proof.
- **`break_even_relative_weight_%`**: minimum relative weight required for a proof to break even under the configured economics.


In [27]:

def render_preset_comparison(
    missed_epochs: int = 1,
    eth_usd: float = DEFAULT_ETH_USD,
    gas_price_gwei: float = DEFAULT_GAS_PRICE_GWEI,
    aztc_usd: float = DEFAULT_AZTC_USD,
    infra_cost_per_proof_usd: float = DEFAULT_INFRA_COST_PER_PROOF_USD,
    gas_units_per_proof: int = DEFAULT_GAS_UNITS_PER_PROOF,
    epoch_reward_pool_aztc: float = DEFAULT_EPOCH_REWARD_POOL_AZTC,
    active_full_weight_provers: int = DEFAULT_ACTIVE_FULL_WEIGHT_PROVERS,
):
    summary = compare_presets_table(
        missed_epochs=missed_epochs,
        eth_usd=eth_usd,
        gas_price_gwei=gas_price_gwei,
        aztc_usd=aztc_usd,
        infra_cost_per_proof_usd=infra_cost_per_proof_usd,
        gas_units_per_proof=gas_units_per_proof,
        epoch_reward_pool_aztc=epoch_reward_pool_aztc,
        active_full_weight_provers=active_full_weight_provers,
    )

    display(Markdown("### Side-by-side recovery and cost summary"))
    display(format_summary_table(summary))

    paths = {}
    for name, params in PARAMETER_PRESETS.items():
        df = recovery_dataframe(params, missed_epochs=missed_epochs)
        econ = add_economics(
            df,
            params,
            eth_usd=eth_usd,
            gas_price_gwei=gas_price_gwei,
            aztc_usd=aztc_usd,
            infra_cost_per_proof_usd=infra_cost_per_proof_usd,
            gas_units_per_proof=gas_units_per_proof,
            epoch_reward_pool_aztc=epoch_reward_pool_aztc,
            active_full_weight_provers=active_full_weight_provers,
        )
        paths[name] = econ[econ["event"] == "recovery_proof"].copy()

    fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
    (recovery_ax, cash_ax), (penalty_ax, profit_ax) = axes
    fig.suptitle(f"Recovery and cost summary after {missed_epochs} missed epoch(s)", fontsize=16)

    for name, path in paths.items():
        recovery_ax.plot(path["proof_index_after_return"], 100 * path["relative_weight"], marker="o", linewidth=2, label=name)
    recovery_ax.axhline(100, linestyle="--", linewidth=1, label="full weight")
    recovery_ax.set_xlabel("Successful proofs after return")
    recovery_ax.set_ylabel("Relative reward weight (%)")
    recovery_ax.set_title("Recovery path")
    recovery_ax.legend(fontsize=9)
    recovery_ax.grid(True, alpha=0.3)

    for name, path in paths.items():
        cash_ax.plot(path["proof_index_after_return"], path["cumulative_cash_spend_usd"], marker="o", linewidth=2, label=f"{name}: cash spend")
    cash_ax.set_xlabel("Successful proofs after return")
    cash_ax.set_ylabel("Cumulative gross recovery cash spend (USD)")
    cash_ax.set_title("Gross cash spend to reach full weight")
    cash_ax.legend(fontsize=9)
    cash_ax.grid(True, alpha=0.3)

    miss_range = np.arange(1, max(10, missed_epochs) + 1)
    for name, params in PARAMETER_PRESETS.items():
        first_return_shares = []
        for m in miss_range:
            path = recovery_dataframe(params, missed_epochs=int(m))
            first_proof = path[path["event"] == "recovery_proof"].iloc[0]
            first_return_shares.append(100 * first_proof["share"] / params.k)
        penalty_ax.plot(miss_range, first_return_shares, marker="o", linewidth=2, label=name)
    penalty_ax.set_xlabel("Consecutive missed epochs before returning")
    penalty_ax.set_ylabel("Share on first return proof (%)")
    penalty_ax.set_title("Penalty cliff")
    penalty_ax.legend(fontsize=9)
    penalty_ax.grid(True, alpha=0.3)

    gas_grid = np.linspace(0, max(1.0, gas_price_gwei * 3), 100)
    full_weight_reward_usd = epoch_reward_pool_aztc * aztc_usd / max(active_full_weight_provers, 1)
    profits = [
        full_weight_reward_usd - cash_cost_per_proof_usd(gas_units_per_proof, g, eth_usd, infra_cost_per_proof_usd)
        for g in gas_grid
    ]
    profit_ax.plot(gas_grid, profits, linewidth=2)
    profit_ax.axhline(0, linestyle="--", linewidth=1, label="break-even")
    profit_ax.axvline(gas_price_gwei, linestyle=":", linewidth=2, label="configured gas price")
    profit_ax.set_xlabel("Gas price (gwei)")
    profit_ax.set_ylabel("Full-weight profit per proof (USD)")
    profit_ax.set_title("Full-weight profitability")
    profit_ax.legend(fontsize=9)
    profit_ax.grid(True, alpha=0.3)

    plt.show()


if WIDGETS_AVAILABLE:
    comparison_controls = widgets.VBox([
        widgets.HBox([
            widgets.IntSlider(value=1, min=1, max=20, step=1, description="Missed epochs"),
            widgets.IntSlider(value=DEFAULT_ACTIVE_FULL_WEIGHT_PROVERS, min=2, max=200, step=1, description="Active provers"),
        ]),
        widgets.HBox([
            widgets.FloatText(value=DEFAULT_EPOCH_REWARD_POOL_AZTC, description="Epoch reward AZTC"),
            widgets.FloatText(value=DEFAULT_AZTC_USD, description="AZTC/USD"),
            widgets.FloatText(value=DEFAULT_ETH_USD, description="ETH/USD"),
        ]),
        widgets.HBox([
            widgets.FloatText(value=DEFAULT_GAS_PRICE_GWEI, description="Gas gwei"),
            widgets.IntText(value=DEFAULT_GAS_UNITS_PER_PROOF, description="Gas units/proof"),
            widgets.FloatText(value=DEFAULT_INFRA_COST_PER_PROOF_USD, description="Infra $/proof"),
        ]),
    ])

    controls_flat = {
        "missed_epochs": comparison_controls.children[0].children[0],
        "active_full_weight_provers": comparison_controls.children[0].children[1],
        "epoch_reward_pool_aztc": comparison_controls.children[1].children[0],
        "aztc_usd": comparison_controls.children[1].children[1],
        "eth_usd": comparison_controls.children[1].children[2],
        "gas_price_gwei": comparison_controls.children[2].children[0],
        "gas_units_per_proof": comparison_controls.children[2].children[1],
        "infra_cost_per_proof_usd": comparison_controls.children[2].children[2],
    }

    out = widgets.interactive_output(render_preset_comparison, controls_flat)
    display(comparison_controls, out)
else:
    render_preset_comparison()


Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<IPython.core.display.Markdown object>…


## Interactive single-parameter-set simulator

Use this section to inspect one preset or define a custom parameter set. This is the best view for understanding exactly where the profitability threshold intersects the reward curve.


In [28]:

def get_params_from_inputs(
    preset: str,
    increment: int,
    max_score: int,
    a: int,
    minimum: int,
    k: int,
    decay_rate: int,
) -> RewardBoosterParams:
    if preset != "Custom":
        return PARAMETER_PRESETS[preset]
    return RewardBoosterParams(
        name="Custom",
        increment=increment,
        max_score=max_score,
        a=a,
        minimum=minimum,
        k=k,
        decay_rate=decay_rate,
    )


def render_single_scenario(
    preset: str = "AZIP-5 proposed",
    increment: int = 101_400,
    max_score: int = 367_500,
    a: int = 250_000,
    minimum: int = 10_000,
    k: int = 1_000_000,
    decay_rate: int = 100_000,
    missed_epochs: int = 1,
    eth_usd: float = DEFAULT_ETH_USD,
    gas_price_gwei: float = DEFAULT_GAS_PRICE_GWEI,
    aztc_usd: float = DEFAULT_AZTC_USD,
    infra_cost_per_proof_usd: float = DEFAULT_INFRA_COST_PER_PROOF_USD,
    gas_units_per_proof: int = DEFAULT_GAS_UNITS_PER_PROOF,
    epoch_reward_pool_aztc: float = DEFAULT_EPOCH_REWARD_POOL_AZTC,
    active_full_weight_provers: int = DEFAULT_ACTIVE_FULL_WEIGHT_PROVERS,
):
    params = get_params_from_inputs(preset, increment, max_score, a, minimum, k, decay_rate)
    summary = summarize_scenario(
        params,
        missed_epochs=missed_epochs,
        eth_usd=eth_usd,
        gas_price_gwei=gas_price_gwei,
        aztc_usd=aztc_usd,
        infra_cost_per_proof_usd=infra_cost_per_proof_usd,
        gas_units_per_proof=gas_units_per_proof,
        epoch_reward_pool_aztc=epoch_reward_pool_aztc,
        active_full_weight_provers=active_full_weight_provers,
    )
    df = recovery_dataframe(params, missed_epochs=missed_epochs)
    econ = add_economics(
        df,
        params,
        eth_usd=eth_usd,
        gas_price_gwei=gas_price_gwei,
        aztc_usd=aztc_usd,
        infra_cost_per_proof_usd=infra_cost_per_proof_usd,
        gas_units_per_proof=gas_units_per_proof,
        epoch_reward_pool_aztc=epoch_reward_pool_aztc,
        active_full_weight_provers=active_full_weight_provers,
    )
    proofs = econ[econ["event"] == "recovery_proof"].copy()

    cash_per_proof = summary["cash_cost_per_proof_usd"]
    full_reward_usd = summary["full_weight_reward_per_proof_usd"]
    share_required, rel_required, breakeven_status = break_even_share(
        params,
        cash_per_proof,
        epoch_reward_pool_aztc,
        aztc_usd,
        active_full_weight_provers,
    )
    required_score = score_for_share(share_required, params)

    summary_display = pd.DataFrame([summary]).T.rename(columns={0: "value"})
    display(Markdown(f"### Scenario summary: {params.name}"))
    display(summary_display)

    gross_cash = proofs["cash_spend_usd"].sum()
    reward_loss = econ["reward_opportunity_loss_usd"].sum()
    display(Markdown(
        f"""
**Headline:** after **{missed_epochs} missed epoch(s)**, this parameter set needs **{len(proofs):,} successful proof(s)** to reach the first full-weight proof.  
Gross recovery cash spend: **${gross_cash:,.2f}**. Reward opportunity loss: **${reward_loss:,.2f}**.  
Full-weight reward/proof: **${full_reward_usd:,.2f}**. Cash cost/proof: **${cash_per_proof:,.2f}**.  
Break-even status: **{breakeven_status}**
"""
    ))

    display(Markdown("### Recovery path table"))
    path_cols = [
        "event",
        "timeline_epoch",
        "proof_index_after_return",
        "score",
        "relative_weight",
        "reward_usd",
        "cash_spend_usd",
        "profit_usd",
        "reward_opportunity_loss_usd",
        "cumulative_cash_spend_usd",
        "cumulative_reward_opportunity_loss_usd",
    ]
    display(econ[path_cols].head(50))
    if len(econ) > 50:
        display(Markdown(f"Showing first 50 of {len(econ):,} rows. The full dataframe is available as `econ`."))

    score_grid = np.linspace(0, params.max_score, 500)
    share_grid = share_from_score(score_grid, params)

    miss_range = np.arange(1, 21)
    first_return_profit = []
    first_return_weight = []
    for m in miss_range:
        d = recovery_dataframe(params, missed_epochs=int(m))
        e = add_economics(
            d,
            params,
            eth_usd=eth_usd,
            gas_price_gwei=gas_price_gwei,
            aztc_usd=aztc_usd,
            infra_cost_per_proof_usd=infra_cost_per_proof_usd,
            gas_units_per_proof=gas_units_per_proof,
            epoch_reward_pool_aztc=epoch_reward_pool_aztc,
            active_full_weight_provers=active_full_weight_provers,
        )
        p0 = e[e["event"] == "recovery_proof"].iloc[0]
        first_return_profit.append(p0["profit_usd"])
        first_return_weight.append(100 * p0["relative_weight"])

    fig, axes = plt.subplots(2, 3, figsize=(21, 11), constrained_layout=True)
    curve_ax, recovery_ax, per_proof_ax = axes[0]
    cost_ax, first_weight_ax, first_profit_ax = axes[1]
    fig.suptitle(f"{params.name}: single-parameter recovery panels", fontsize=16)

    curve_ax.plot(score_grid / params.max_score, 100 * share_grid / params.k, linewidth=2, label="share curve")
    if np.isfinite(rel_required):
        curve_ax.axhline(100 * rel_required, linestyle="--", linewidth=2, label="break-even relative weight")
        if np.isfinite(required_score):
            curve_ax.axvline(required_score / params.max_score, linestyle=":", linewidth=2, label="break-even score")
    curve_ax.scatter(proofs["score"] / params.max_score, 100 * proofs["relative_weight"], s=40, label="recovery proofs")
    curve_ax.set_xlabel("Score / maxScore")
    curve_ax.set_ylabel("Relative weight (%)")
    curve_ax.set_title("Reward curve and profitability threshold")
    curve_ax.legend(fontsize=8)
    curve_ax.grid(True, alpha=0.3)

    recovery_ax.plot(proofs["proof_index_after_return"], 100 * proofs["relative_weight"], marker="o", linewidth=2, label="relative weight (%)")
    recovery_ax.axhline(100, linestyle="--", linewidth=1, label="full weight")
    if np.isfinite(rel_required):
        recovery_ax.axhline(100 * rel_required, linestyle=":", linewidth=2, label="break-even relative weight")
    recovery_ax.set_xlabel("Successful proofs after return")
    recovery_ax.set_ylabel("Relative weight (%)")
    recovery_ax.set_title("Recovery to full reward weight")
    recovery_ax.legend(fontsize=8)
    recovery_ax.grid(True, alpha=0.3)

    per_proof_ax.plot(proofs["proof_index_after_return"], proofs["reward_usd"], marker="o", linewidth=2, label="reward per proof")
    per_proof_ax.axhline(cash_per_proof, linestyle="--", linewidth=2, label="cash cost per proof")
    per_proof_ax.plot(proofs["proof_index_after_return"], proofs["profit_usd"], marker="x", linewidth=2, label="profit per proof")
    per_proof_ax.axhline(0, linestyle=":", linewidth=1, label="break-even profit")
    per_proof_ax.set_xlabel("Successful proofs after return")
    per_proof_ax.set_ylabel("USD")
    per_proof_ax.set_title("Revenue, cost, and profit during recovery")
    per_proof_ax.legend(fontsize=8)
    per_proof_ax.grid(True, alpha=0.3)

    cost_ax.plot(econ["timeline_epoch"], econ["cumulative_cash_spend_usd"], marker="o", linewidth=2, label="cumulative gross cash spend")
    cost_ax.plot(econ["timeline_epoch"], econ["cumulative_reward_opportunity_loss_usd"], marker="x", linewidth=2, label="cumulative reward opportunity loss")
    cost_ax.set_xlabel("Timeline epoch after the last healthy proof")
    cost_ax.set_ylabel("USD")
    cost_ax.set_title("Cumulative cost to return to full weight")
    cost_ax.legend(fontsize=8)
    cost_ax.grid(True, alpha=0.3)

    first_weight_ax.plot(miss_range, first_return_weight, marker="o", linewidth=2, label="first-return weight")
    if np.isfinite(rel_required):
        first_weight_ax.axhline(100 * rel_required, linestyle="--", linewidth=2, label="break-even relative weight")
    first_weight_ax.set_xlabel("Consecutive missed epochs")
    first_weight_ax.set_ylabel("First-return relative weight (%)")
    first_weight_ax.set_title("First-return weight after consecutive misses")
    first_weight_ax.legend(fontsize=8)
    first_weight_ax.grid(True, alpha=0.3)

    first_profit_ax.plot(miss_range, first_return_profit, marker="o", linewidth=2, label="first-return profit")
    first_profit_ax.axhline(0, linestyle="--", linewidth=1, label="break-even profit")
    first_profit_ax.set_xlabel("Consecutive missed epochs")
    first_profit_ax.set_ylabel("First-return proof profit (USD)")
    first_profit_ax.set_title("First-return proof profitability after consecutive misses")
    first_profit_ax.legend(fontsize=8)
    first_profit_ax.grid(True, alpha=0.3)

    plt.show()

    return econ


if WIDGETS_AVAILABLE:
    preset_dropdown = widgets.Dropdown(
        options=list(PARAMETER_PRESETS.keys()) + ["Custom"],
        value="AZIP-5 proposed",
        description="Preset",
        layout=widgets.Layout(width="350px"),
    )

    custom_controls = widgets.VBox([
        widgets.HTML("<b>Custom parameters used only when Preset = Custom</b>"),
        widgets.HBox([
            widgets.IntText(value=101_400, description="increment"),
            widgets.IntText(value=367_500, description="maxScore"),
            widgets.IntText(value=250_000, description="a"),
        ]),
        widgets.HBox([
            widgets.IntText(value=10_000, description="minimum"),
            widgets.IntText(value=1_000_000, description="k"),
            widgets.IntText(value=100_000, description="decayRate"),
        ]),
    ])

    single_controls = widgets.VBox([
        widgets.HBox([
            preset_dropdown,
            widgets.IntSlider(value=1, min=1, max=20, step=1, description="Missed epochs"),
            widgets.IntSlider(value=DEFAULT_ACTIVE_FULL_WEIGHT_PROVERS, min=2, max=200, step=1, description="Active provers"),
        ]),
        widgets.HBox([
            widgets.FloatText(value=DEFAULT_EPOCH_REWARD_POOL_AZTC, description="Epoch reward AZTC"),
            widgets.FloatText(value=DEFAULT_AZTC_USD, description="AZTC/USD"),
            widgets.FloatText(value=DEFAULT_ETH_USD, description="ETH/USD"),
        ]),
        widgets.HBox([
            widgets.FloatText(value=DEFAULT_GAS_PRICE_GWEI, description="Gas gwei"),
            widgets.IntText(value=DEFAULT_GAS_UNITS_PER_PROOF, description="Gas units/proof"),
            widgets.FloatText(value=DEFAULT_INFRA_COST_PER_PROOF_USD, description="Infra $/proof"),
        ]),
        custom_controls,
    ])

    single_controls_flat = {
        "preset": preset_dropdown,
        "missed_epochs": single_controls.children[0].children[1],
        "active_full_weight_provers": single_controls.children[0].children[2],
        "epoch_reward_pool_aztc": single_controls.children[1].children[0],
        "aztc_usd": single_controls.children[1].children[1],
        "eth_usd": single_controls.children[1].children[2],
        "gas_price_gwei": single_controls.children[2].children[0],
        "gas_units_per_proof": single_controls.children[2].children[1],
        "infra_cost_per_proof_usd": single_controls.children[2].children[2],
        "increment": custom_controls.children[1].children[0],
        "max_score": custom_controls.children[1].children[1],
        "a": custom_controls.children[1].children[2],
        "minimum": custom_controls.children[2].children[0],
        "k": custom_controls.children[2].children[1],
        "decay_rate": custom_controls.children[2].children[2],
    }

    single_out = widgets.interactive_output(render_single_scenario, single_controls_flat)
    display(single_controls, single_out)
else:
    econ = render_single_scenario()


Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<IPython.core.display.Markdown object>…


## Gas and infrastructure sensitivity tables

This section creates small sensitivity tables you can use for decision-making. It uses the selected/default gas units and infrastructure assumptions but sweeps gas price and ETH/USD.


In [29]:

def gas_sensitivity_table(
    gas_units_per_proof: float = DEFAULT_GAS_UNITS_PER_PROOF,
    infra_cost_per_proof_usd: float = DEFAULT_INFRA_COST_PER_PROOF_USD,
    gas_prices_gwei=(1, 5, 10, 20, 50),
    eth_prices_usd=(2_000, 3_000, 4_000, 5_000),
) -> pd.DataFrame:
    rows = []
    for gas_gwei in gas_prices_gwei:
        for eth_usd in eth_prices_usd:
            gas_usd = gas_cost_usd(gas_units_per_proof, gas_gwei, eth_usd)
            rows.append({
                "gas_price_gwei": gas_gwei,
                "ETH_USD": eth_usd,
                "gas_cost_per_proof_usd": gas_usd,
                "infra_cost_per_proof_usd": infra_cost_per_proof_usd,
                "cash_cost_per_proof_usd": gas_usd + infra_cost_per_proof_usd,
            })
    return pd.DataFrame(rows)


def recovery_cash_sensitivity_table(
    missed_epochs: int = 1,
    gas_units_per_proof: float = DEFAULT_GAS_UNITS_PER_PROOF,
    infra_cost_per_proof_usd: float = DEFAULT_INFRA_COST_PER_PROOF_USD,
    gas_prices_gwei=(1, 5, 10, 20, 50),
    eth_prices_usd=(2_000, 3_000, 4_000, 5_000),
) -> pd.DataFrame:
    rows = []
    for preset_name, params in PARAMETER_PRESETS.items():
        proofs_to_full = len(recovery_dataframe(params, missed_epochs=missed_epochs).query("event == 'recovery_proof'"))
        for gas_gwei in gas_prices_gwei:
            for eth_usd in eth_prices_usd:
                cash = cash_cost_per_proof_usd(gas_units_per_proof, gas_gwei, eth_usd, infra_cost_per_proof_usd)
                rows.append({
                    "parameter_set": preset_name,
                    "missed_epochs": missed_epochs,
                    "recovery_proofs_to_full_weight": proofs_to_full,
                    "gas_price_gwei": gas_gwei,
                    "ETH_USD": eth_usd,
                    "cash_cost_per_proof_usd": cash,
                    "gross_recovery_cash_spend_usd": proofs_to_full * cash,
                })
    return pd.DataFrame(rows)

print("Gas + infra cost per proof:")
display(gas_sensitivity_table())

print("Gross recovery cash spend after one missed epoch:")
display(recovery_cash_sensitivity_table(missed_epochs=1))


Gas + infra cost per proof:


,gas_price_gwei,ETH_USD,gas_cost_per_proof_usd,infra_cost_per_proof_usd,cash_cost_per_proof_usd
0,1,2000,7.700000,78.000000,85.700000
1,1,3000,11.550000,78.000000,89.550000
2,1,4000,15.400000,78.000000,93.400000
3,1,5000,19.250000,78.000000,97.250000
4,5,2000,38.500000,78.000000,116.500000
5,5,3000,57.750000,78.000000,135.750000
6,5,4000,77.000000,78.000000,155.000000
7,5,5000,96.250000,78.000000,174.250000
8,10,2000,77.000000,78.000000,155.000000
9,10,3000,115.500000,78.000000,193.500000


Gross recovery cash spend after one missed epoch:


,parameter_set,missed_epochs,recovery_proofs_to_full_weight,gas_price_gwei,ETH_USD,cash_cost_per_proof_usd,gross_recovery_cash_spend_usd
0,Current / original,1,4,1,2000,85.700000,342.800000
1,Current / original,1,4,1,3000,89.550000,358.200000
2,Current / original,1,4,1,4000,93.400000,373.600000
3,Current / original,1,4,1,5000,97.250000,389.000000
4,Current / original,1,4,5,2000,116.500000,466.000000
5,Current / original,1,4,5,3000,135.750000,543.000000
6,Current / original,1,4,5,4000,155.000000,620.000000
7,Current / original,1,4,5,5000,174.250000,697.000000
8,Current / original,1,4,10,2000,155.000000,620.000000
9,Current / original,1,4,10,3000,193.500000,774.000000
